In [2]:
from pathlib import Path

import yaml

PROJECT_ROOT = Path.cwd()
with open(PROJECT_ROOT / "config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

paths = config["paths"]
embedding_config = config["embedding"]
DATA_PATH = PROJECT_ROOT / paths["bertopic_ready"]
EMBEDDING_PATH = PROJECT_ROOT / paths["embeddings"]

In [3]:
# Dependencies are managed by pyproject.toml / uv.lock for local runs.


In [4]:
# If running elsewhere, install the project dependencies before executing this notebook.


In [5]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

In [6]:
df = pd.read_csv(DATA_PATH)


In [7]:
df.head()

df.shape

(35993, 18)

In [8]:
df

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,word_count,language
0,2025-09-18T02:30:29.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"To Whom It May Concern, I am writing in respon...",NaN,"EQUIFAX, INC.",NY,109XX,NaN,Web,2025-09-18T02:30:55.000Z,Closed with explanation,Yes,16011810.0,439,Language.ENGLISH
1,2026-03-13T00:27:38.000Z,Checking or savings account,Checking account,Problem with a lender or other company chargin...,Transaction was not authorized,"on Monday, [PROTECTED]/[PROTECTED]/[PROTECTED]...",NaN,"Block, Inc.",OR,97317,NaN,Web,2026-03-13T00:46:10.000Z,Closed with explanation,Yes,20223337.0,144,Language.ENGLISH
2,2025-05-23T22:58:46.000Z,"Payday loan, title loan, personal loan, or adv...",Payday loan,Money was taken from your bank account on the ...,NaN,I am writing to bring to your attention an iss...,Company believes the complaint provided an opp...,Albert Corporation,CT,06405,NaN,Web,2025-05-23T23:27:55.000Z,Closed with explanation,Yes,13691637.0,266,Language.ENGLISH
3,2025-07-26T23:50:05.000Z,Checking or savings account,Checking account,Problem with a lender or other company chargin...,Transaction was not authorized,some people got hold of debit card information...,NaN,"HUNTINGTON NATIONAL BANK, THE",MN,554XX,Older American,Web,2025-07-27T00:49:31.000Z,Closed with monetary relief,Yes,14903535.0,99,Language.ENGLISH
4,2024-12-26T20:40:18.000Z,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,Good day! I am reaching out regarding several ...,NaN,"EQUIFAX, INC.",CT,06830,NaN,Web,2024-12-26T20:40:41.000Z,Closed with non-monetary relief,Yes,11285148.0,292,Language.ENGLISH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35988,2025-06-18T21:40:05.000Z,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,Experian Credit Services [PROTECTED] [PROTECTE...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NY,105XX,NaN,Web,2025-06-18T21:42:38.000Z,Closed with explanation,Yes,14147519.0,138,Language.ENGLISH
35989,2023-11-07T18:18:13.000Z,Credit reporting or other personal consumer re...,Credit reporting,Credit monitoring or identity theft protection...,Problem canceling credit monitoring or identif...,i got a copy of my credit repor not to long a ...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,TX,77016,NaN,Web,2023-11-07T18:36:29.000Z,Closed with explanation,Yes,7822102.0,69,Language.ENGLISH
35990,2024-06-18T21:31:06.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Someone opened invalid accounts using my infor...,NaN,CAPITAL ONE FINANCIAL CORPORATION,NV,89122,NaN,Web,2024-06-18T21:31:07.000Z,Closed with explanation,Yes,9287994.0,7,Language.ENGLISH
35991,2025-10-09T18:56:15.000Z,"Money transfer, virtual currency, or money ser...",Mobile or digital wallet,"Managing, opening, or closing your mobile wall...",NaN,[PROTECTED] closed my account after I sent dri...,NaN,"Paypal Holdings, Inc",TX,77099,NaN,Web,2025-10-09T19:05:37.000Z,Closed with explanation,Yes,16471267.0,9,Language.ENGLISH


In [9]:
documents = df["Consumer complaint narrative"].tolist()

In [10]:
len(documents)



35993

In [11]:
type(documents)



list

In [12]:
documents[0]

'To Whom It May Concern, I am writing in response to a received on regarding an alleged debt [PROTECTED] CREDIT CARD Collection you claim I owe. Please be advised that I am requesting validation of this debt, as is my right under the Fair Debt CREDIT CARD Collection Practices Act, 15 U.S.C. 1692g. Pursuant to this law, I respectfully request that you provide the following information : 1. The name and address of the original creditor. 2. A detailed statement of the alleged debt, including the amount and how it was calculated. 3. A copy of the original signed debt collection agreement or contract. 4. Proof that you are legally authorized to collect this debt. 5. A full accounting of payments, interest, fees, and charges applied to the alleged balance. 6. A copy of the judgment ( if any ) or other documents proving your legal right to collect this debt. Until you provide proper validation, I am requesting that all collection activity cease, including any reporting to the credit bureaus. 

In [13]:
import torch

print(torch.cuda.is_available())

print(torch.cuda.get_device_name(0))

True
Tesla T4


In [14]:
from sentence_transformers import SentenceTransformer

In [42]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    embedding_config["model_name"],
    model_kwargs={
        "torch_dtype": getattr(torch, embedding_config["torch_dtype"]),
        "device_map": embedding_config["device_map"]
    }
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [43]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

cuda


In [44]:
model = model.to(device)

In [45]:
test_embedding = model.encode(
    documents[0],
    convert_to_numpy=True
)

print(test_embedding.shape)

(2048,)


In [46]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [47]:
!nvidia-smi

Sat Jul 11 06:24:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             34W /   70W |   10761MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [48]:
embeddings = model.encode_document(
    documents,
    batch_size=embedding_config["batch_size"],
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/17997 [00:00<?, ?it/s]

In [49]:
import numpy as np

np.save(EMBEDDING_PATH, embeddings)


In [50]:
loaded = np.load(EMBEDDING_PATH)

print(loaded.shape)


(35993, 2048)


In [52]:
print(f"Embeddings saved to: {EMBEDDING_PATH}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
2+2

In [ ]:
from sentence_transformers import util

sentences = [
    "I cannot access my bank account",
    "My online banking is not working",
    "I love playing football"
]

emb = model.encode(sentences)

similarity = util.cos_sim(emb, emb)

print(similarity)